# Skill suggestion

**Difficulty:** Intermediate

Pick at most one skill for an agent turn out of a large roster, in two requests: one `Choice` ranks every skill from its short index description plus three `Noul` gates asking whether a skill is even needed; a second, narrower `Choice` re-reads only the top few with fuller descriptions, and can reject all of them.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/skill_suggestion (click to expand)</summary>

# Skill suggestion

> Picks at most one skill for an agent turn out of the 182 in Nous Research's Hermes catalog: one TypeSafe request ranks every skill and asks whether the turn needs one at all, a second reads the top three properly and can reject all of them. The winner's name goes into a single line of the agent's system prompt, and both the wrong skills it loads and the ones it loads when nothing fits drop by more than half.

An agent with a large skill roster makes its choice on almost no information — the roster reaches it as an index, one line per skill, description truncated to 60 characters so it doesn't crowd the conversation. At that width, a skill that *edits* `.pptx` files reads nearly the same as one that *authors* them. Ask for a pitch deck and the agent may load the wrong one; on a turn where nothing fits, a list of names still invites a guess.

Two TypeSafe requests go in front of the agent's own decision:

- **Call 1 — skim the whole roster.** One `Choice` over every skill name (criteria = the same short index description the agent gets) ranks the roster; three `Noul` questions ask, in different ways, whether the turn wants an *action taken* rather than an explanation given. Their mean, oriented so "yes" means "needs a skill," decides whether to suggest anything at all — below `0.30`, nothing is suggested.
- **Call 2 — read the top 3 properly.** A narrower `Choice`, now with each skill's *full* description (not the truncated index line), free to answer "none of these actually fit."

The winner's name goes into one extra line of the agent's system prompt for that turn — the agent keeps its full roster and its own judgment; the line just says which entry to look at first.

## The result, over 488 measured turns against `claude-haiku-4-5`

|                                       | loads the wrong skill | loads one when nothing fits |
| ------------------------------------- | ---------------------- | ---------------------------- |
| agent alone, with just its roster     | 16.8%                  | 9.8%                          |
| **agent with a TypeSafe suggestion**  | **7.3%**               | **4.0%**                     |
| agent handed the right answer         | 2.5%                   | 1.2%                          |

Wrong first picks landed in the *right skill's own category* far more often than chance — the hard part was telling a few lookalikes apart, not searching blindly. The third row is the floor: even handed the exactly correct skill name, the agent still doesn't always load it.

## Worked example: a pitch-deck request

Two lookalike skills exist on the test roster: `powerpoint` (edit an existing deck) and `pptx-author` (build one from scratch, headless). On a request to build a deck skeleton, the wide call — working only from 60-character descriptions — ranks `powerpoint` at `0.700` and `pptx-author` at `0.300`, backwards for a request that's clearly about *authoring*, not editing. Only the narrow call, reading each skill's full description, has enough to tell them apart.

> The full cookbook runs both calls against a real 182-skill Hermes roster and measures an actual Claude agent's tool calls across 488 requests — trimmed here to the two-call mechanism, since running a second agent to measure isn't something this repo does. Read the full cookbook at [docs.typesafe.ai/cookbooks/skill_suggestion](https://docs.typesafe.ai/cookbooks/skill_suggestion) any time.

</details>

## The core idea

This is re-ranking (`cookbooks/how-to/re-ranking.ipynb`) with a twist: instead of scoring every candidate independently with the same amount of information, the *first* pass deliberately works from less information (short descriptions, cheap to hold all at once) and the *second* pass spends more tokens on fewer candidates (full descriptions, only for the top few). Cheap-then-expensive, not expensive-everywhere. The three gate `Noul`s riding along in call 1 are the same "ask more, it's nearly free" fan-out logic as always — they decide *whether to suggest at all*, independently of *what* to suggest.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice, noul } from "npm:@typesafe-ai/sdk";

// A tiny roster, short index-style descriptions - as an agent would actually see them.
const roster: Record<string, string> = {
  "pdf-editor": "Edit, merge, split PDF files.",
  "powerpoint": "Create, read, edit .pptx decks, slides, notes.",
  "pptx-author": "Build PowerPoint decks headless with python-pptx.",
  "apple-notes": "Manage Apple Notes via CLI: create, search, edit.",
  "web-search": "Search the web and return result snippets.",
};

const request =
  "Can you put together a pitch deck skeleton (cover, market overview, comps) as a .pptx, " +
  "using our firm template, with each number footnoted back to where it came from?";

// Call 1: rank the whole roster + ask whether a skill is needed at all, one request.
const wideResponse = await client.systemOne({
  state: { request },
  questions: {
    which: choice(
      "Which of these skills, if any, is the right one to load to help with the user's latest request?",
      roster,
    ),
    actsOnSystem: noul(
      "Is the assistant being asked to act on the user's files, accounts, devices, or online services, rather than only to explain or advise?",
    ),
    followsProcedure: noul(
      "Would a careful expert answering this consult a specific documented procedure or set of commands, rather than answering from general understanding?",
    ),
    proseSuffices: noul(
      "Could a knowledgeable generalist fully satisfy this request in prose, with no tools, no documentation, and no access to the user's files or accounts?",
    ),
  },
});

const { which, actsOnSystem, followsProcedure, proseSuffices } = wideResponse.answers;
const gate = (actsOnSystem.noul + followsProcedure.noul + (1 - proseSuffices.noul)) / 3;

const ranked = Object.entries(which.probabilities).sort((a, b) => b[1] - a[1]);
console.log("needs a skill:", gate.toFixed(2), gate >= 0.3 ? "-> suggest" : "-> stay quiet");
console.log("top 3 (wide, short descriptions):", ranked.slice(0, 3));

// Call 2: re-rank just the top 3, now with full descriptions - and a "none" escape hatch.
const fullDescriptions: Record<string, string> = {
  "powerpoint":
    "Create, read, and edit PowerPoint (.pptx) files: decks, slides, speaker notes, and existing templates. Best for opening and modifying an existing deck.",
  "pptx-author":
    "Author new PowerPoint decks from scratch, headless, using python-pptx: build slide layouts, charts, and footnotes programmatically without opening PowerPoint.",
  "apple-notes": "Manage Apple Notes via a memo CLI: create, search, and edit notes.",
};

const top3 = ranked.slice(0, 3).map(([name]) => name);
const narrowCriteria: Record<string, string> = Object.fromEntries(
  top3.map((name) => [name, fullDescriptions[name] ?? roster[name]]),
);
narrowCriteria["none"] = "None of these three actually fit this request.";

const narrowResponse = await client.systemOne({
  state: { request },
  questions: {
    which: choice(
      "Given these fuller descriptions, which of these skills actually fits the request - or none of them?",
      narrowCriteria,
    ),
  },
});

console.log(
  "reranked winner (narrow, full descriptions):",
  narrowResponse.answers.which.choice,
  narrowResponse.answers.which.confidence.toFixed(2),
);

## What just happened

The request is genuinely about *building* a deck from scratch with a template and footnoted figures — closer to `pptx-author` than `powerpoint` once you read the full descriptions, but the two are easy to blur on short ones alone. Compare the wide call's top pick against the narrow call's winner: if they differ, that's the whole cookbook's finding happening in miniature — cheap short descriptions get the right *category*, expensive full descriptions get the right *specific answer*.

**Try it yourself:** rewrite the request to be about *editing an existing* firm-template deck instead of building one from scratch, and predict whether the narrow call now favors `powerpoint` over `pptx-author` — the two skills' full descriptions are written to split exactly on that distinction.

## Related

- `cookbooks/how-to/re-ranking.ipynb` — the same shape (score candidates, sort, take the best), without the cheap-then-expensive two-tier structure
- `lessons/patterns/0006-speculative-fan-out.ipynb` — why the three gate `Noul`s ride along in the same call as the ranking `Choice` instead of a separate request
- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the `0.30` gate threshold is the same idea as a confidence threshold, applied to a composed signal instead of a single `confidence` field

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.